# Dzien 2 / Krok 3: Trening, Optuna i MLflow

**Input:** `checkpoints/02_train.parquet`, `checkpoints/02_test.parquet`

Tematy:
- MLflow: logowanie eksperymentow, porownanie runow, Model Registry
- Optuna: inteligentne przeszukiwanie przestrzeni hiperparametrow
- Siec neuronowa dla porownania z XGBoost

Sciaga: `notebooks/building_blocks_and_examples/optuna.ipynb`


In [ ]:
import pandas as pd
import numpy as np
import os, time
import mlflow
import mlflow.xgboost
import xgboost as xgb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import roc_auc_score, classification_report, roc_curve

TARGET = "is_bad_experience"
df_train = pd.read_parquet("checkpoints/02_train.parquet")
df_test  = pd.read_parquet("checkpoints/02_test.parquet")

FEATURE_COLS = [c for c in df_train.columns if c != TARGET]
X_train, y_train = df_train[FEATURE_COLS], df_train[TARGET]
X_test,  y_test  = df_test[FEATURE_COLS],  df_test[TARGET]

scale_pos_weight = (1 - y_train.mean()) / y_train.mean()
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Cech: {len(FEATURE_COLS)}  |  scale_pos_weight: {scale_pos_weight:.1f}")


## MLflow: polaczenie i konfiguracja eksperymentu

In [ ]:
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5000"))
EXPERIMENT = "olist_bad_review_day2"
mlflow.set_experiment(EXPERIMENT)

print(f"URI:        {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT}")


In [ ]:
def evaluate(model, X_test, y_test, threshold=0.5):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred  = (y_proba >= threshold).astype(int)
    auc     = roc_auc_score(y_test, y_proba)
    report  = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    p1      = report.get("1", {})
    return {
        "auc":         auc,
        "precision_1": p1.get("precision", 0),
        "recall_1":    p1.get("recall", 0),
        "f1_1":        p1.get("f1-score", 0),
    }


## Run 1: baseline z MLflow

MLflow run to jak `git commit` dla eksperymentu - zapisuje parametry, metryki i model.
Potem mozesz je porownac w UI pod http://localhost:5000.

```python
with mlflow.start_run(run_name="nazwa"):
    mlflow.log_params({...})   # hiperparametry
    # ... trening ...
    mlflow.log_metrics({...})  # auc, precision, recall
    mlflow.xgboost.log_model(model, artifact_path="model", ...)
```


In [ ]:
BASE_PARAMS = {
    "objective": "binary:logistic", "eval_metric": "auc",
    "n_estimators": 300, "max_depth": 5, "learning_rate": 0.05,
    "scale_pos_weight": float(scale_pos_weight), "random_state": 42, "n_jobs": -1,
}

# TODO: Wytrenuj model bazowy wewnatrz MLflow run "01_baseline"
# Zaloguj: parametry, metryki z evaluate(), model przez mlflow.xgboost.log_model()
# Zapisz run_id_baseline = mlflow.active_run().info.run_id

with mlflow.start_run(run_name="01_baseline"):
    ...

print(f"Baseline AUC: {metrics['auc']:.4f}")


## Run 2: RandomSearch

Klasyczne podejscie: losowe probkowanie przestrzeni hiperparametrow.
RandomizedSearchCV - odpowiednik brute-force: sprawdza N losowych kombinacji.


In [ ]:
%%time
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    "n_estimators":  randint(100, 500),
    "max_depth":     randint(3, 10),
    "learning_rate": uniform(0.01, 0.29),
    "subsample":     uniform(0.5, 0.5),
}

# TODO: Uruchom RandomizedSearchCV wewnatrz MLflow run "02_random_search"
# Uzyj n_iter=20, cv=3, scoring="roc_auc"
# Zmierz czas (time.time()), zaloguj search_time_sec
# Retrenuj najlepszy model na pelnym train, zaloguj model

with mlflow.start_run(run_name="02_random_search"):
    ___

print(f"RandomSearch  AUC: {metrics_rs['auc']:.4f}  czas: {t_rs:.0f}s")
print(f"Najlepsze params: {rs.best_params_}")


## Run 3: Optuna

Optuna uzywa TPE (Tree-structured Parzen Estimator) - uczy sie z poprzednich prob
i szuka w obszarach, ktore rokuja lepiej.

Sciaga: `notebooks/building_blocks_and_examples/optuna.ipynb`


In [ ]:
%%time
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# TODO: Uruchom Optuna wewnatrz MLflow run "03_optuna"
# objective(trial) powinno:
#   - dobrac parametry przez trial.suggest_int / trial.suggest_float
#   - wytrenowac model, obliczyc AUC
#   - zalogowac mlflow.log_metric("trial_auc", auc, step=trial.number)
#   - zwrocic AUC
# Po optimize: retrenuj finalny model z study.best_params, zaloguj do tego samego runu

with mlflow.start_run(run_name="03_optuna") as optuna_run:
    ___

print(f"Optuna    AUC: {study.best_value:.4f}  czas: {t_opt:.0f}s")
print(f"Najlepsze params: {study.best_params}")


In [ ]:
# Wykres przebiegu optymalizacji (gotowe)
trial_vals = [t.value for t in study.trials]
best_so_far = pd.Series(trial_vals).cummax()

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["AUC per proba", "Waznosc hiperparametrow"])
fig.add_trace(go.Scatter(y=trial_vals, mode="markers", name="Kazda proba",
                         marker=dict(color=trial_vals, colorscale="Viridis", size=6)),
              row=1, col=1)
fig.add_trace(go.Scatter(y=best_so_far.tolist(), mode="lines", name="Najlepszy dotad",
                         line=dict(color="crimson", width=2)), row=1, col=1)

importance = optuna.importance.get_param_importances(study)
fig.add_trace(go.Bar(x=list(importance.values()), y=list(importance.keys()),
                     orientation="h", marker_color="steelblue"), row=1, col=2)
fig.update_layout(height=430, title="Optuna: przebieg optymalizacji")
fig.show()


## Run 4: Prosta siec neuronowa (Keras)

XGBoost na danych tabelarycznych zwykle wygrywa z sieciami neuronowymi.
Ale warto zobaczyc, dlaczego sieci sa trudniejsze w obsludze:

| | XGBoost | Siec neuronowa |
|---|---|---|
| Skalowanie cech | nie potrzeba | **wymagane** |
| Niezbalansowane klasy | `scale_pos_weight` | `class_weight` w `.fit()` |
| Overfitting | regularyzacja wbudowana | Dropout + EarlyStopping |
| Training curve | nie ma | **kluczowa** - widac overfitting |
| Czas treningu | szybki | wolniejszy na tabelarycznych |


In [ ]:
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Skalowanie wymagane - gradient descent wrazliwy na skale wejsc
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

class_weights_arr = compute_class_weight(
    "balanced", classes=np.array([0, 1]), y=y_train
)
class_weight_dict = {0: float(class_weights_arr[0]), 1: float(class_weights_arr[1])}

print(f"Srednia po skalowaniu:  {X_train_scaled.mean():.4f}  (powinno byc ~0)")
print(f"Odch. std po skalowaniu:{X_train_scaled.std():.4f}  (powinno byc ~1)")
print(f"class_weight: {class_weight_dict}")


In [ ]:
tf.random.set_seed(42)

# TODO: Zdefiniuj architekture sieci neuronowej (keras.Sequential)
# Architektura piramidy: Input -> Dense(128, relu) -> Dropout(0.3)
#                              -> Dense(64, relu) -> Dropout(0.3)
#                              -> Dense(32, relu) -> Dense(1, sigmoid)
# Dropout(0.3): podczas treningu losowo wylacza 30% neuronow -> lepsza generalizacja
model_nn = keras.Sequential([
    # TODO
], name="olist_bad_review_nn")

# TODO: Skompiluj model z Adam(lr=0.001), binary_crossentropy, AUC jako metryke
model_nn.compile(
    # TODO
)

model_nn.summary()


In [ ]:
%%time

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_auc", patience=10, restore_best_weights=True, mode="max"
)

# TODO: Wytrenuj model wewnatrz MLflow run "04_neural_network"
# Parametry treningu: epochs=100, batch_size=512, class_weight=class_weight_dict
# Zaloguj metryki per epoka (train_loss, train_auc, val_loss, val_auc) przez step=epoch
# - to daje krzywa uczenia w MLflow UI
# Na koncu zaloguj model przez mlflow.tensorflow.log_model(...)

with mlflow.start_run(run_name="04_neural_network") as nn_run:
    ___

print(f"NN  AUC: {metrics_nn['auc']:.4f}  epok: {len(history.history['loss'])}")


In [ ]:
# Krzywa uczenia (gotowe - wazniejsza niz AUC dla sieci)
epochs_range = list(range(len(history.history["loss"])))

fig = make_subplots(rows=1, cols=2, subplot_titles=["AUC", "Loss"])
fig.add_trace(go.Scatter(x=epochs_range, y=history.history["auc"],
                         name="train", line=dict(color="steelblue")), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs_range, y=history.history["val_auc"],
                         name="val", line=dict(color="steelblue", dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs_range, y=history.history["loss"],
                         name="train", line=dict(color="crimson")), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs_range, y=history.history["val_loss"],
                         name="val", line=dict(color="crimson", dash="dash")), row=1, col=2)
fig.update_xaxes(title_text="Epoka")
fig.update_layout(height=380, title="Training curve: siec neuronowa")
fig.show()
print("  train vs val blisko siebie  -> dobra generalizacja")
print("  train rosnie, val spada     -> overfitting")


## Porownanie runow

In [ ]:
comparison = pd.DataFrame([
    {"Model": "Baseline",      **metrics},
    {"Model": "RandomSearch",  **metrics_rs},
    {"Model": "Optuna",        **metrics_opt},
    {"Model": "NeuralNetwork", **metrics_nn},
]).set_index("Model")

print(comparison.round(4).to_string())

fig = go.Figure()
for col, color in [("auc","steelblue"),("recall_1","crimson"),("precision_1","seagreen")]:
    fig.add_trace(go.Bar(name=col, x=comparison.index,
                         y=comparison[col].astype(float), marker_color=color))
fig.update_layout(barmode="group", title="Porownanie modeli",
                  yaxis_range=[0.5, 1.0], height=420)
fig.show()


## Model Registry: zarejestruj najlepszy model

In [ ]:
from mlflow.tracking import MlflowClient

MODEL_NAME = "olist_bad_review"
client = MlflowClient()

# TODO: Zarejestruj najlepszy model (Optuna) w Model Registry
# mlflow.register_model(model_uri=f"runs:/{run_id_opt}/model", name=MODEL_NAME)
# Ustaw alias "champion" na zarejestrowana wersje
result = ___

print(f"Zarejestrowano: {MODEL_NAME} v{result.version}")
